# Restrições numéricas em campos

Vimos que alguns tipos especializados do Pydantic, como `PositiveInt`, especificam não só o tipo numérico, mas restrições adicionais.

`PositiveInt` é ótimo para inteiros positivos, mas não há um tipo especializado para, por exemplo, um inteiro maior que `1`.

A classe `Field` oferece opções mais flexíveis: parâmetros que aplicam restrições específicas.

In [1]:
from pydantic import BaseModel, Field, ValidationError

Para restrições numéricas, temos:
- `gt` -> `>`
- `ge` -> `>=`
- `lt` -> `<`
- `le` -> `<=`
- `multiple_of` -> o número precisa ser múltiplo deste valor

O uso é bem direto:

In [2]:
class Model(BaseModel):
    number: float = Field(gt=2, le=10, multiple_of=2)

In [3]:
print(f"Model.model_fields: {Model.model_fields!r}")

Model.model_fields: {'number': FieldInfo(annotation=float, required=True, metadata=[Gt(gt=2), Le(le=10), MultipleOf(multiple_of=2)])}


In [4]:
m = Model(number=4)
print(f"m: {m!r}")

m: Model(number=4.0)


In [5]:
try:
    Model(number=4.2)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
number
  Input should be a multiple of 2 [type=multiple_of, input_value=4.2, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/multiple_of


In [6]:
try:
    Model(number=3)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
number
  Input should be a multiple of 2 [type=multiple_of, input_value=3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/multiple_of


In [7]:
try:
    Model(number=14)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
number
  Input should be less than or equal to 10 [type=less_than_equal, input_value=14, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


Então `PositiveInt` é, na prática, o mesmo que `Field(gt=0)`.

Na verdade, já tínhamos visto isso ao inspecionar os campos do modelo.

In [8]:
class Model(BaseModel):
    number: int = Field(gt=0)

In [9]:
print(f"Model.model_fields: {Model.model_fields!r}")

Model.model_fields: {'number': FieldInfo(annotation=int, required=True, metadata=[Gt(gt=0)])}


In [10]:
from pydantic import PositiveInt

class Model(BaseModel):
    number: PositiveInt

In [11]:
print(f"Model.model_fields: {Model.model_fields!r}")

Model.model_fields: {'number': FieldInfo(annotation=int, required=True, metadata=[Gt(gt=0)])}


In [12]:
class Model(BaseModel):
    number: float

## Infinito e NaN

Floats em Python podem ser `nan` (*not a number*), `inf` e `-inf`. Por padrão, um campo `float` aceita esses valores — o que costuma ser
surpresa desagradável em cálculos e ao serializar para JSON (que não tem infinito nem NaN).

In [13]:
print(f"float('nan'), float('inf'), float('-inf'): {float('nan'), float('inf'), float('-inf')!r}")

float('nan'), float('inf'), float('-inf'): (nan, inf, -inf)


In [14]:
print(f"Model(number=float('inf')), Model(number=float('-inf')): {Model(number=float('inf')), Model(number=float('-inf'))!r}")

Model(number=float('inf')), Model(number=float('-inf')): (Model(number=inf), Model(number=-inf))


Para proibir, use `allow_inf_nan=False`:

In [15]:
class Model(BaseModel):
    number: float = Field(allow_inf_nan=False)

In [16]:
try:
    Model(number=float('inf'))
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
number
  Input should be a finite number [type=finite_number, input_value=inf, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/finite_number


In [17]:
try:
    Model(number=float('nan'))
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
number
  Input should be a finite number [type=finite_number, input_value=nan, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/finite_number


> 💡 **Complemento:** por que isso importa na serialização — o JSON gerado pelo Pydantic com `inf` não é JSON válido segundo a
> especificação (o padrão do Pydantic é emitir `null`; dá para mudar com `ser_json_inf_nan`):

In [18]:
class Loose(BaseModel):
    number: float

print(f'Loose(number=float("inf")).model_dump_json(): {Loose(number=float("inf")).model_dump_json()!r}')

Loose(number=float("inf")).model_dump_json(): '{"number":null}'
